In [2]:
import pandas as pd
import numpy as np

# **Tratamento de Dados**

## Diágnóstico Inicial: Conhecendo o Dataset.

In [103]:
url = 'https://raw.githubusercontent.com/Kaio0809/Dataset_Filmes/refs/heads/main/dados/filmes.csv'
filmes = pd.read_csv('dados/filmes.csv',index_col=False )

In [104]:
print(filmes.shape)
print(filmes.info())

(4755, 19)
<class 'pandas.DataFrame'>
RangeIndex: 4755 entries, 0 to 4754
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_tmdb           4755 non-null   int64  
 1   titulo            4755 non-null   str    
 2   titulo_original   4755 non-null   str    
 3   data_lancamento   4755 non-null   str    
 4   ano               4755 non-null   int64  
 5   generos           4755 non-null   str    
 6   genero_ids        4732 non-null   str    
 7   nota              4755 non-null   float64
 8   popularidade      4755 non-null   float64
 9   orcamento_usd     4755 non-null   float64
 10  bilheteria_usd    4755 non-null   float64
 11  duracao_min       4755 non-null   float64
 12  paises_producao   4754 non-null   str    
 13  codigos_paises    4755 non-null   str    
 14  idioma_original   4755 non-null   str    
 15  diretores         4694 non-null   str    
 16  elenco_principal  4608 non-null   str    


In [105]:
categoricas = ['generos','paises_producao','codigos_paises','idioma_original','diretores','elenco_principal']
numericas = ['ano','nota','numero_votos','popularidade','orcamento_usd','bilheteria_usd','duracao_min','']

In [106]:
for coluna in categoricas:
    print(f'Valores {coluna}: {filmes[coluna].unique()}')

Valores generos: <StringArray>
[                     'Fantasia, Terror, Comédia',
                                   'Documentário',
                                          'Drama',
                    'Animação, Comédia, Fantasia',
             'Ficção científica, Terror, Comédia',
                  'Ação, Drama, História, Guerra',
                          'Documentário, Família',
                      'Crime, Fantasia, Mistério',
                           'Documentário, Música',
                               'Romance, Comédia',
 ...
                       'Faroeste, Comédia, Crime',
            'Terror, Ficção científica, Aventura',
              'Ficção científica, Mistério, Ação',
                         'Terror, Thriller, Ação',
                              'Romance, Thriller',
            'Animação, Fantasia, Aventura, Drama',
 'Animação, Fantasia, Aventura, Família, Comédia',
           'Comédia, Aventura, Família, Animação',
              'Faroeste, Ação, Aventura, Roman

In [107]:
f = filmes.groupby('ano')['id_tmdb'].count()
for x,y in f.items():
    print(f'Ano: {x} - Count: {y}')

Ano: 1900 - Count: 24
Ano: 1901 - Count: 27
Ano: 1902 - Count: 25
Ano: 1903 - Count: 21
Ano: 1904 - Count: 26
Ano: 1905 - Count: 25
Ano: 1906 - Count: 26
Ano: 1907 - Count: 25
Ano: 1908 - Count: 26
Ano: 1909 - Count: 27
Ano: 1910 - Count: 27
Ano: 1911 - Count: 26
Ano: 1912 - Count: 25
Ano: 1913 - Count: 29
Ano: 1914 - Count: 32
Ano: 1915 - Count: 27
Ano: 1916 - Count: 27
Ano: 1917 - Count: 26
Ano: 1918 - Count: 27
Ano: 1919 - Count: 27
Ano: 1920 - Count: 26
Ano: 1921 - Count: 25
Ano: 1922 - Count: 25
Ano: 1923 - Count: 28
Ano: 1924 - Count: 26
Ano: 1925 - Count: 25
Ano: 1926 - Count: 28
Ano: 1927 - Count: 26
Ano: 1928 - Count: 28
Ano: 1929 - Count: 25
Ano: 1930 - Count: 24
Ano: 1931 - Count: 26
Ano: 1932 - Count: 25
Ano: 1933 - Count: 22
Ano: 1934 - Count: 23
Ano: 1935 - Count: 21
Ano: 1936 - Count: 23
Ano: 1937 - Count: 25
Ano: 1938 - Count: 25
Ano: 1939 - Count: 24
Ano: 1940 - Count: 25
Ano: 1941 - Count: 23
Ano: 1942 - Count: 25
Ano: 1943 - Count: 24
Ano: 1944 - Count: 22
Ano: 1945 

## Tratando Valores Nulos!

In [80]:
filmes = filmes.replace(['False', '-'], None, inplace=True)
filmes.to_csv('dados/filmes.csv', index=False)

In [110]:
count = 0
for col in filmes.columns:
    for i in range(filmes.shape[0]):
        if pd.isnull(filmes.loc[i, col]):
            count += 1

print(f'Total: {count}')

Total: 765


In [111]:
# Transformar valores zerados na média dos dados reais.

m = lambda df, col: np.mean([df.loc[i, col] for i in range(df.shape[0]) if df.loc[i, col] != 0])
media_orcamento = m(filmes, 'orcamento_usd')
media_bilheteria = m(filmes, 'bilheteria_usd')

print(f'Média Orçamento: {media_orcamento}')
print(f'Média Bilheteria: {media_bilheteria}')

filmes['orcamento_usd'] = filmes['orcamento_usd'].replace(0, media_orcamento)
filmes['bilheteria_usd'] = filmes['bilheteria_usd'].replace(0, media_bilheteria)

filmes['duracao_min'] = filmes['duracao_min'].astype(float)
medias_duracao = filmes.groupby('ano')['duracao_min'].transform(
    lambda x: x[x != 0].mean()
)

filmes.loc[filmes['duracao_min'] == 0, 'duracao_min'] = medias_duracao
filmes.to_csv('dados/filmes.csv', index=False)

Média Orçamento: 42648034.58045792
Média Bilheteria: 162410435.06583175


## Criação de Colunas Úteis

In [112]:
# Criando Lucros
filmes['lucro'] = filmes['bilheteria_usd'] - filmes['orcamento_usd']

## Excluir colunas Inuteis

In [113]:
filmes = filmes.drop(columns=['genero_ids','idioma_original'])

## Separação dos Datasets

In [114]:
colunas_generos = ['id_tmdb','generos']
colunas_paises = ['id_tmdb','paises_producao','codigos_paises']
colunas_filmes = ['id_tmdb','titulo','titulo_original','data_lancamento','ano','nota','popularidade','orcamento_usd','bilheteria_usd','lucro','duracao_min','codigos_paises','diretores','elenco_principal','sinopse','poster_url']


generos = filmes[colunas_generos]
paises = filmes[colunas_paises]
filmes = filmes[colunas_filmes]

In [116]:
generos['generos'] = generos['generos'].str.split(', ')

generos = generos.explode('generos').reset_index(drop=True)


In [ ]:

paises['paises_producao'] = paises['paises_producao'].str.split(', ')
paises['codigos_paises'] = paises['codigos_paises'].str.split(', ')

paises = paises.explode(['paises_producao', 'codigos_paises'])



In [120]:
paises.to_csv('dados/paises.csv', index=False)

In [121]:
generos

,id_tmdb,generos
0,104700,Fantasia
1,104700,Terror
2,104700,Comédia
3,1679884,Documentário
4,195626,Drama
...,...,...
11909,1437696,História
11910,1506560,Cinema TV
11911,1506560,Família
11912,1506560,Ação


In [123]:
generos.to_csv('dados/generos.csv')

## Atualização do Resultado



In [124]:
filmes.to_csv('dados/cinetimeline.csv')

In [125]:
print(filmes.info())
print(generos.info())
print(paises.info())

<class 'pandas.DataFrame'>
RangeIndex: 4755 entries, 0 to 4754
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_tmdb           4755 non-null   int64  
 1   titulo            4755 non-null   str    
 2   titulo_original   4755 non-null   str    
 3   data_lancamento   4755 non-null   str    
 4   ano               4755 non-null   int64  
 5   nota              4755 non-null   float64
 6   popularidade      4755 non-null   float64
 7   orcamento_usd     4755 non-null   float64
 8   bilheteria_usd    4755 non-null   float64
 9   lucro             4755 non-null   float64
 10  duracao_min       4755 non-null   float64
 11  codigos_paises    4755 non-null   str    
 12  diretores         4694 non-null   str    
 13  elenco_principal  4608 non-null   str    
 14  sinopse           4315 non-null   str    
 15  poster_url        4662 non-null   str    
dtypes: float64(6), int64(2), str(8)
memory usage: 594.5 K

In [4]:
movies = pd.read_csv('dados/progresso_pais_ano.csv')
movies

,ano,codigo_pais,pais,quantidade_total
0,1900,AD,Andorra,0
1,1900,AE,United Arab Emirates,0
2,1900,AF,Afghanistan,0
3,1900,AG,Antigua and Barbuda,0
4,1900,AI,Anguilla,0
...,...,...,...,...
32938,2022,NaN,Namibia,0
32939,2023,NaN,Namibia,11
32940,2024,NaN,Namibia,7
32941,2025,NaN,Namibia,3


In [8]:
abc = movies.groupby('ano')['quantidade_total'].sum()
for ano,qtd in abc.items():
    print(f"{ano} - {qtd} - 10%: {0.1*qtd}")

1900 - 480 - 10%: 48.0
1901 - 364 - 10%: 36.4
1902 - 309 - 10%: 30.900000000000002
1903 - 427 - 10%: 42.7
1904 - 349 - 10%: 34.9
1905 - 325 - 10%: 32.5
1906 - 383 - 10%: 38.300000000000004
1907 - 391 - 10%: 39.1
1908 - 519 - 10%: 51.900000000000006
1909 - 625 - 10%: 62.5
1910 - 975 - 10%: 97.5
1911 - 1232 - 10%: 123.2
1912 - 1690 - 10%: 169.0
1913 - 1699 - 10%: 169.9
1914 - 1734 - 10%: 173.4
1915 - 1846 - 10%: 184.60000000000002
1916 - 1922 - 10%: 192.20000000000002
1917 - 1842 - 10%: 184.20000000000002
1918 - 1634 - 10%: 163.4
1919 - 1714 - 10%: 171.4
1920 - 1711 - 10%: 171.10000000000002
1921 - 1635 - 10%: 163.5
1922 - 1566 - 10%: 156.60000000000002
1923 - 1448 - 10%: 144.8
1924 - 1583 - 10%: 158.3
1925 - 1701 - 10%: 170.10000000000002
1926 - 1868 - 10%: 186.8
1927 - 1895 - 10%: 189.5
1928 - 1953 - 10%: 195.3
1929 - 1964 - 10%: 196.4
1930 - 2103 - 10%: 210.3
1931 - 2010 - 10%: 201.0
1932 - 2088 - 10%: 208.8
1933 - 2074 - 10%: 207.4
1934 - 2135 - 10%: 213.5
1935 - 2275 - 10%: 227.5
19